# OQIM AI — Deep Q-Network (DQN) Versiyasi v2.0

**Yangilangan versiya.** Barcha kamchiliklar tuzatildi:
- ✅ Epsilon decay tezlashtirildi (500 epizodda 0.01 ga yetadi)
- ✅ Simulyatsiya uzaytirildi (60 → 200 qadam)
- ✅ Reward funksiyasi yaxshilandi (balans hisobga olinadi)
- ✅ Reset holatlari real chorrahaga yaqinlashtirildi
- ✅ Epsilon kamayish grafigi qo'shildi
- ✅ DBSCAN bog'lanishi tushuntirildi

---
## Bo'lim 1: Kutubxonalar va Sozlamalar

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from collections import deque
import random
import copy

# === SOZLAMALAR (v2.0 — optimallashtirilgan) ===
EPISODES      = 500
BATCH_SIZE    = 64
GAMMA         = 0.99
EPSILON_START = 1.0
EPSILON_END   = 0.01
EPSILON_DECAY = 0.99    # TUZATILDI: 0.995 → 0.99 (endi 460-epizodda 0.01 ga yetadi)
LEARNING_RATE = 0.0005
MEMORY_SIZE   = 10000   # TUZATILDI: 5000 → 10000 (ko'proq tajriba saqlaydi)
TARGET_UPDATE = 10

print("✅ Bo'lim 1 tayyor: Kutubxonalar yuklandi!")
print(f"PyTorch versiyasi: {torch.__version__}")
print(f"\nEpsilon 0.01 ga taxminan {int(np.log(0.01/1.0)/np.log(0.99))}-epizodda yetadi")

---
## Bo'lim 2: Chorraha Muhiti (Environment) — Yaxshilangan

**Tuzatildi:**
- `max_steps`: 60 → 200 (uzun simulyatsiya = yaxshiroq strategiya)
- `reset()`: 0-10 → 0-25 (real chorrahaga yaqinroq)
- `reward`: balans jarima qo'shildi (bitta tomonda navbat to'planmasligi uchun)

In [ ]:
class Intersection:
    """
    4 tomonlama chorraha simulyatori (v2.0).
    - ns_queue: Shimol-Janub yo'lidagi kutayotgan mashinalar
    - ew_queue: Sharq-G'arb yo'lidagi kutayotgan mashinalar
    """
    def __init__(self):
        self.ns_queue = 0
        self.ew_queue = 0
        self.step_count = 0
        self.max_steps = 200  # TUZATILDI: 60 → 200
    
    def get_state(self):
        return np.array([
            self.ns_queue / 50.0,
            self.ew_queue / 50.0,
            (self.step_count % 10) / 10.0
        ], dtype=np.float32)
    
    def step(self, action):
        """
        action=0: NS ga qisqa yashil (15 sek) — 8 ta o'tadi
        action=1: EW ga qisqa yashil (15 sek) — 8 ta o'tadi
        action=2: NS ga uzun yashil (30 sek) — 15 ta o'tadi
        action=3: EW ga uzun yashil (30 sek) — 15 ta o'tadi
        """
        ns_incoming = np.random.poisson(8)
        ew_incoming = np.random.poisson(6)
        self.ns_queue += ns_incoming
        self.ew_queue += ew_incoming
        
        if action == 0:
            ns_pass, ew_pass = 8, 0
        elif action == 1:
            ns_pass, ew_pass = 0, 8
        elif action == 2:
            ns_pass, ew_pass = 15, 0
        else:
            ns_pass, ew_pass = 0, 15
        
        self.ns_queue = max(0, self.ns_queue - ns_pass)
        self.ew_queue = max(0, self.ew_queue - ew_pass)
        self.step_count += 1
        
        # TUZATILDI: Reward yaxshilandi — balans jarima qo'shildi
        total = self.ns_queue + self.ew_queue
        imbalance = abs(self.ns_queue - self.ew_queue)
        reward = 10.0 - total * 0.5 - imbalance * 0.3
        
        done = self.step_count >= self.max_steps
        return self.get_state(), reward, done
    
    def reset(self):
        # TUZATILDI: 0-10 → 0-25 (real chorrahaga yaqinroq)
        self.ns_queue = random.randint(0, 25)
        self.ew_queue = random.randint(0, 25)
        self.step_count = 0
        return self.get_state()

print("✅ Bo'lim 2 tayyor: Yaxshilangan chorraha muhiti yaratildi!")
print(f"   max_steps={200}, reset_range=0-25, reward=10-total*0.5-imbalance*0.3")

---
## Bo'lim 3: Neyron Tarmoq (DQN) + Xotira

In [ ]:
class DQN(nn.Module):
    """Deep Q-Network. Kirish: 3 (NS, EW, vaqt). Chiqish: 4 harakat."""
    def __init__(self, state_size=3, action_size=4):
        super(DQN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(state_size, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_size)
        )
    
    def forward(self, x):
        return self.network(x)


class ReplayMemory:
    """AI ning xotirasi. O'tgan tajribalarni saqlaydi."""
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        batch = random.sample(self.memory, batch_size)
        states = np.array([b[0] for b in batch])
        actions = [b[1] for b in batch]
        rewards = [b[2] for b in batch]
        next_states = np.array([b[3] for b in batch])
        dones = [b[4] for b in batch]
        return states, actions, rewards, next_states, dones
    
    def __len__(self):
        return len(self.memory)

print("✅ Bo'lim 3 tayyor: Neyron tarmoq arxitekturasi qurildi!")
print(f"Model tuzilishi:\n{DQN()}")

---
## Bo'lim 4: AI ni O'qitish (Training)

**v2.0 yangiliklar:** Epsilon tezroq kamayadi, epsilon tarixi saqlanadi.

In [ ]:
model = DQN()
target_model = copy.deepcopy(model)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
memory = ReplayMemory(MEMORY_SIZE)
env = Intersection()
epsilon = EPSILON_START

reward_history = []
loss_history = []
epsilon_history = []  # YANGI: Epsilon kamayishini kuzatish

print("AI o'qitilmoqda... (Bu 30-90 soniya davom etadi)\n")

for episode in range(EPISODES):
    state = env.reset()
    total_reward = 0
    ep_losses = []
    
    while True:
        if random.random() < epsilon:
            action = random.randint(0, 3)
        else:
            with torch.no_grad():
                q_values = model(torch.FloatTensor(state))
                action = q_values.argmax().item()
        
        next_state, reward, done = env.step(action)
        memory.push(state, action, reward, next_state, done)
        total_reward += reward
        state = next_state
        
        if len(memory) >= BATCH_SIZE:
            states, actions, rewards, next_states, dones = memory.sample(BATCH_SIZE)
            states_t = torch.FloatTensor(states)
            actions_t = torch.LongTensor(actions)
            rewards_t = torch.FloatTensor(rewards)
            next_states_t = torch.FloatTensor(next_states)
            dones_t = torch.FloatTensor(dones)
            
            current_q = model(states_t).gather(1, actions_t.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                next_q = target_model(next_states_t).max(1)[0]
            target_q = rewards_t + GAMMA * next_q * (1 - dones_t)
            
            loss = nn.SmoothL1Loss()(current_q, target_q)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_losses.append(loss.item())
        
        if done:
            break
    
    if (episode + 1) % TARGET_UPDATE == 0:
        target_model.load_state_dict(model.state_dict())
    
    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
    epsilon_history.append(epsilon)
    reward_history.append(total_reward)
    if ep_losses:
        loss_history.append(np.mean(ep_losses))
    
    if (episode + 1) % 100 == 0:
        avg = np.mean(reward_history[-50:])
        print(f"Epizod {episode+1}/{EPISODES} | Baho: {avg:.1f} | Epsilon: {epsilon:.4f}")

print(f"\n✅ Bo'lim 4 tayyor: AI o'qitildi!")
print(f"Boshlang'ich: {np.mean(reward_history[:20]):.1f} -> Yakuniy: {np.mean(reward_history[-20:]):.1f}")
print(f"Yakuniy epsilon: {epsilon:.4f} (maqsad: 0.01)")

---
## Bo'lim 5: Natijalar va Isbot

1. Reward ↑, Loss ↓, **Epsilon ↓ grafiklarini** ko'rsatamiz
2. Uch bosqichli taqqoslash
3. DBSCAN tirbandlik tahlili

In [ ]:
# === A. O'QITISH GRAFIKLARI (3 ta) ===
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# Reward
ax1.plot(reward_history, color='blue', alpha=0.2)
w = 30
smoothed = [np.mean(reward_history[max(0,i-w):i+1]) for i in range(len(reward_history))]
ax1.plot(smoothed, color='blue', linewidth=2.5, label="O'rtacha baho")
ax1.set_title('Reward — Vaqt bilan oshadi')
ax1.set_xlabel('Epizod')
ax1.set_ylabel('Umumiy Baho')
ax1.legend()
ax1.grid(True)

# Loss
if loss_history:
    ax2.plot(loss_history, color='red', alpha=0.2)
    sl = [np.mean(loss_history[max(0,i-30):i+1]) for i in range(len(loss_history))]
    ax2.plot(sl, color='red', linewidth=2.5, label="O'rtacha xatolik")
    ax2.set_title('Loss — Vaqt bilan kamayadi')
    ax2.set_xlabel('Epizod')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)

# YANGI: Epsilon kamayish grafigi
ax3.plot(epsilon_history, color='green', linewidth=2.5)
ax3.axhline(y=EPSILON_END, color='red', linestyle='--', label=f'Maqsad: {EPSILON_END}')
ax3.set_title('Epsilon — Tasodifiylik kamayadi')
ax3.set_xlabel('Epizod')
ax3.set_ylabel('Epsilon')
ax3.legend()
ax3.grid(True)

plt.tight_layout()
plt.show()

# Epsilon statistikasi
reached_idx = next((i for i, e in enumerate(epsilon_history) if e <= EPSILON_END + 0.001), None)
if reached_idx:
    print(f"\n✅ Epsilon {EPSILON_END} ga {reached_idx+1}-epizodda yetdi!")
else:
    print(f"\n⚠️ Epsilon {EPSILON_END} ga yetmadi. Yakuniy: {epsilon_history[-1]:.4f}")

In [ ]:
# === B. UCH BOSQICHLI TAQQOSLASH ===
test_steps = 120

# 1. Hozirgi holat (fiks svetafor)
np.random.seed(42)
random.seed(42)
env_curr = Intersection()
env_curr.reset()
h_current = []
for i in range(test_steps):
    action_fixed = 0 if i % 2 == 0 else 1
    env_curr.step(action_fixed)
    h_current.append(env_curr.ns_queue + env_curr.ew_queue)

# 2. AI holat (DQN)
np.random.seed(42)
random.seed(42)
env_ai = Intersection()
state = env_ai.reset()
h_ai = []
for _ in range(test_steps):
    with torch.no_grad():
        q_vals = model(torch.FloatTensor(state))
        action = q_vals.argmax().item()
    state, _, _ = env_ai.step(action)
    h_ai.append(env_ai.ns_queue + env_ai.ew_queue)

# 3. Kelajak (V2X)
np.random.seed(42)
random.seed(42)
env_fut = Intersection()
state = env_fut.reset()
h_future = []
for _ in range(test_steps):
    with torch.no_grad():
        q_vals = model(torch.FloatTensor(state))
        action = q_vals.argmax().item()
    env_fut.ns_queue = int(env_fut.ns_queue * 0.3)
    env_fut.ew_queue = int(env_fut.ew_queue * 0.3)
    state, _, _ = env_fut.step(action)
    h_future.append(env_fut.ns_queue + env_fut.ew_queue)

plt.figure(figsize=(12, 6))
plt.plot(h_current, label='1. Hozirgi holat (Fiks svetafor)', color='red', linewidth=2.5)
plt.plot(h_ai, label='2. AI (DQN v2.0)', color='blue', linewidth=2.5)
plt.plot(h_future, label='3. Kelajak (AI + V2X)', color='green', linewidth=2.5, linestyle='--')
plt.title('Chorrahada Kutayotgan Mashinalar (3 Bosqichli Taqqoslash)')
plt.xlabel('Vaqt (Tsikllar)')
plt.ylabel('Kutayotgan Mashinalar Soni')
plt.legend(loc='upper left')
plt.grid(True)
plt.fill_between(range(test_steps), h_current, h_ai, color='red', alpha=0.08)
plt.fill_between(range(test_steps), h_ai, h_future, color='blue', alpha=0.08)
plt.show()

avg_curr = np.mean(h_current)
avg_ai = np.mean(h_ai)
avg_fut = np.mean(h_future)
print(f"\n🔴 Hozirgi: o'rtacha {avg_curr:.1f} ta mashina kutadi")
print(f"🔵 AI (DQN v2.0): o'rtacha {avg_ai:.1f} ta mashina kutadi")
print(f"🟢 Kelajak (V2X): o'rtacha {avg_fut:.1f} ta mashina kutadi")
if avg_curr > 0:
    print(f"\n🚀 AI samaradorligi: Tirbandlik {100-(avg_ai/avg_curr*100):.1f}% kamaydi")
    print(f"🚀 Kelajak tizimi: Tirbandlik {100-(avg_fut/avg_curr*100):.1f}% kamaydi")

In [ ]:
# === C. DBSCAN — Tirbandlik Zonalarini Aniqlash ===
# TUSHUNTIRISH: DBSCAN va DQN parallel ishlaydi:
# - DBSCAN mashinalar joylashuviga qarab tirbandlik zonalarini aniqlaydi
# - DQN esa shu zonalardagi svetaforlarni optimal boshqaradi
# - Real tizimda: DBSCAN → zona topadi → DQN ga "shu joyda zichlik yuqori" deb ma'lumot beradi

np.random.seed(0)
c1 = np.random.normal([30, 30], 5, (100, 2))
c2 = np.random.normal([70, 70], 4, (100, 2))
noise = np.random.uniform(0, 100, (100, 2))
vehicles = np.vstack((c1, c2, noise))
dbscan = DBSCAN(eps=6, min_samples=8)
labels = dbscan.fit_predict(vehicles)

plt.figure(figsize=(8, 6))
plt.scatter(vehicles[labels == -1, 0], vehicles[labels == -1, 1],
            c='gray', label='Erkin Harakat', alpha=0.5, s=20)
unique = set(labels) - {-1}
clrs = ['red', 'purple', 'orange']
for i, lb in enumerate(unique):
    plt.scatter(vehicles[labels == lb, 0], vehicles[labels == lb, 1],
                c=clrs[i % 3], label=f'Tirbandlik Zonasi {lb+1}', s=60)
plt.title('DBSCAN: Zichlikka Asoslangan Tirbandlik Tahlili')
plt.xlabel('X koordinata (metr)')
plt.ylabel('Y koordinata (metr)')
plt.legend()
plt.grid(True)
plt.show()

n_clusters = len(unique)
n_noise = (labels == -1).sum()
print(f"\n📊 DBSCAN natijalari:")
print(f"   Topilgan tirbandlik zonalari: {n_clusters} ta")
print(f"   Erkin harakatdagi mashinalar: {n_noise} ta")
print(f"   Tirbandlikdagi mashinalar: {len(labels) - n_noise} ta")
print(f"\n✅ Barcha natijalar muvaffaqiyatli ko'rsatildi!")